In [2]:
import pandas as pd
import os
from ossapi import Ossapi, UserLookupKey, ScoreType
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv

In [3]:
load_dotenv()

CLIENT_ID = 48461
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
PLAYER_ID = 11542338
PLAYER = "ZenthicFrost"
CSV_FILE = "osu_dataset.csv"

# Manual Recorder

In [4]:
PHT = timezone(timedelta(hours=8))

def record_latest_play(dataapi, user_id, beatmap_type, played_before):
    # Get the most recent play
    recent_plays = dataapi.user_scores(user_id, type='recent', include_fails=True , limit=10)

    if not recent_plays:
        print("No recent plays found")
        return

    attempt = recent_plays[0]

    # Check if this play ID already exists in the CSV
    if os.path.exists('osu_plays.csv'):
        existing_data = pd.read_csv('osu_plays.csv')
        if attempt.id in existing_data['id'].values:
            print(f"Play ID {attempt.id} is already recorded")
            return

    # Convert timestamps to Philippine Time
    date_started_pht = attempt.started_at.astimezone(PHT) if attempt.started_at else None
    date_ended_pht = attempt.ended_at.astimezone(PHT) if attempt.ended_at else None

    # Fetch full beatmap and beatmapset details
    full_beatmap = dataapi.beatmap(attempt.beatmap_id)
    full_beatmapset = dataapi.beatmapset(full_beatmap.beatmapset_id)

    play = {
        # Beatmapset features
        'song_title': attempt.beatmapset.title if attempt.beatmapset else 'Unknown',
        'artist': attempt.beatmapset.artist if attempt.beatmapset else 'Unknown',
        'genre': full_beatmapset.genre['name'] if full_beatmapset.genre else None,

        # Play metadata
        'id': attempt.id,
        'beatmap_id': attempt.beatmap_id,
        'date_started': date_started_pht,
        'date_ended': date_ended_pht,
        'accuracy': round(attempt.accuracy * 100, 2),   
        'max_combo_player': attempt.max_combo,
        'max_combo_beatmap': full_beatmap.max_combo,
        'is_perfect_combo': attempt.is_perfect_combo,
        'passed': attempt.passed,

        # Statistics
        'max_great': attempt.maximum_statistics.great,
        'great': attempt.statistics.great,
        'ok': attempt.statistics.ok,
        'meh': attempt.statistics.meh,
        'miss': attempt.statistics.miss,

        # Beatmap features
        'ar': attempt.beatmap.ar if attempt.beatmap else None,
        'cs': attempt.beatmap.cs if attempt.beatmap else None,
        'bpm': attempt.beatmap.bpm if attempt.beatmap else None,
        'hit_length': attempt.beatmap.hit_length if attempt.beatmap else None,
        'count_circles': attempt.beatmap.count_circles if attempt.beatmap else None,
        'count_sliders': attempt.beatmap.count_sliders if attempt.beatmap else None,
        'count_spinners': attempt.beatmap.count_spinners if attempt.beatmap else None,
        'difficulty_rating': attempt.beatmap.difficulty_rating if attempt.beatmap else None,
        'beatmap_type' : beatmap_type,
        'played_before' : played_before,
    }

    # Save to CSV
    new_row = pd.DataFrame([play])
    if os.path.exists('osu_plays.csv'):
        new_row.to_csv('osu_plays.csv', mode='a', header=False, index=False)
    else:
        new_row.to_csv('osu_plays.csv', mode='w', header=True, index=False)

    # Print nicely
    print(f"✓ Recorded Play ID {play['id']}:")
    print("="*60)
    print(f"Song: {play['song_title']}")
    print(f"Artist: {play['artist']}")
    print(f"Genre: {play['genre'] if play['genre'] else 'N/A'}")
    print(f"Beatmap ID: {play['beatmap_id']}")
    print(f"Date Started: {play['date_started']}")
    print(f"Date Ended: {play['date_ended']}")
    print(f"Accuracy: {play['accuracy']:.2f}%")
    print(f"Max Combo Player: {play['max_combo_player']}")
    print(f"Max Combo Beatmap: {play['max_combo_beatmap']}")
    print(f"Full Combo?: {play['is_perfect_combo']}")
    print(f"Passed: {play['passed']}")
    print(f"Played Before: {play['played_before']}")
    # ===== Statistics =====
    print()
    print(f"Max Great: {play['max_great']}")
    print(f"Great: {play['great']}")
    print(f"OK: {play['ok']}")
    print(f"Meh: {play['meh']}")
    print(f"Miss: {play['miss']}")
    # ===== Beatmap Features =====
    print()
    print(f"AR: {play['ar']}")
    print(f"CS: {play['cs']}")
    print(f"BPM: {play['bpm'] if play['bpm'] is not None else 'N/A'}")
    print(f"Hit Length (sec): {play['hit_length']}")
    print(f"Count Circles: {play['count_circles']}")
    print(f"Count Sliders: {play['count_sliders']}")
    print(f"Count Spinners: {play['count_spinners']}")
    print(f"Difficulty Rating: {play['difficulty_rating']}")
    print(f"Beatmap Type: {play['beatmap_type']}")
    print("="*60)
    print()

In [5]:
# Beatmap Categories:
# - Jump Maps
# - Stream Maps
# - Tech Maps
# - Hybrid Maps (Jump, Stream, or Tech)
# - Flow (No specific map type)

In [6]:
# Run this to record

beatmap_type = 'Stream'
played_before = True

api = Ossapi(CLIENT_ID, CLIENT_SECRET)
user = api.user("ZenthicFrost", key=UserLookupKey.USERNAME)
record_latest_play(api, user.id, beatmap_type, played_before)

No recent plays found


In [7]:
df = pd.read_csv('osu_plays.csv')
df.tail(5)

,song_title,artist,genre,id,beatmap_id,date_started,date_ended,accuracy,max_combo_player,max_combo_beatmap,...,ar,cs,bpm,hit_length,count_circles,count_sliders,count_spinners,difficulty_rating,beatmap_type,played_before
296,Time Bomb (feat. Veela),Feint & Boyinaband,Electronic,6224127025,1384287,2026-02-15 19:13:58+08:00,2026-02-15 19:16:38+08:00,95.24,712,786,...,9.2,4.0,175.0,144,366,209,0,5.27509,Jump,True
297,TRIPLE PLAY,KASAI HARCORES,Electronic,6224148747,1477910,2026-02-15 19:18:42+08:00,2026-02-15 19:22:12+08:00,88.51,292,1770,...,9.5,3.8,175.0,238,493,565,0,5.94715,Tech,True
298,Playing With Fire,kors k,Video Game,6224158842,2011421,2026-02-15 19:22:58+08:00,2026-02-15 19:24:45+08:00,83.94,220,669,...,9.4,4.0,160.0,92,294,171,4,5.83337,Tech,True
299,The Pretender,Infected Mushroom,Rock,6224198872,221777,2026-02-15 19:28:23+08:00,2026-02-15 19:34:38+08:00,94.42,521,2388,...,9.0,4.0,174.0,364,854,593,3,5.78374,Hybrid,True
300,The Words I Never Said,Mage,Electronic,6224369676,3018109,2026-02-15 20:10:56+08:00,2026-02-15 20:16:03+08:00,84.30,736,2376,...,9.2,3.9,172.0,295,1332,404,6,5.88379,Stream,True


In [8]:
df['beatmap_type'].value_counts()

beatmap_type
Jump      82
Hybrid    79
Stream    71
Tech      69
Name: count, dtype: int64